# Check Input Examples

Use small invented examples to check the input calculations.

Open this notebook in `notebooks/M3/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

## Create home A

In [ ]:
import pandas as pd
import numpy as np
start = pd.Timestamp("2013-01-01", tz="UTC")
times = pd.date_range(start, periods=50, freq="h")
home_a = pd.DataFrame({"LCLid": "A", "hour": times, "kwh": np.arange(50) / 10})
home_a.loc[26, "kwh"] = np.nan
home_a.loc[10, "kwh"] = 0.0
home_a.head()

## Remove hour 3 and add home B

In [ ]:
home_a = home_a.drop(index=3)
home_b = pd.DataFrame({"LCLid": "B", "hour": times[:30], "kwh": 100.0 + np.arange(30)})
hourly = pd.concat([home_a, home_b], ignore_index=True)
hourly.head()

## List the expected hours

In [ ]:
hours = pd.date_range(hourly["hour"].min(), hourly["hour"].max(), freq="h")
hours[:5]

## Make a household-and-hour grid

In [ ]:
homes = sorted(hourly["LCLid"].unique())
grid = pd.MultiIndex.from_product([homes, hours], names=["LCLid", "target_hour"])
len(grid)

## Index the observed values

In [ ]:
values = hourly.set_index(["LCLid", "hour"])["kwh"]
values.index.names = ["LCLid", "target_hour"]
values.head()

## Include absent hours

In [ ]:
values = values.reindex(grid)
values.head()

## Add the target

In [ ]:
all_rows = values.reset_index(name="target_kwh")
all_rows.head()

## Find the previous hour

In [ ]:
previous_time = all_rows["target_hour"] - pd.Timedelta(hours=1)
keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], previous_time])
previous_values = values.reindex(keys)
all_rows["lag_1h_kwh"] = previous_values.values
all_rows.head()

## Find the same hour yesterday

In [ ]:
yesterday_time = all_rows["target_hour"] - pd.Timedelta(hours=24)
keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], yesterday_time])
yesterday_values = values.reindex(keys)
all_rows["lag_24h_kwh"] = yesterday_values.values
all_rows.iloc[22:27]

## Add calendar information

In [ ]:
all_rows["hour"] = all_rows["target_hour"].dt.hour
all_rows["weekday"] = all_rows["target_hour"].dt.dayofweek
all_rows.iloc[22:27]

## Find rows with missing values

In [ ]:
all_rows["row_status"] = "kept"
all_rows.loc[all_rows["lag_24h_kwh"].isna(), "row_status"] = "missing_lag_24h"
all_rows.loc[all_rows["lag_1h_kwh"].isna(), "row_status"] = "missing_lag_1h"
all_rows.loc[all_rows["target_kwh"].isna(), "row_status"] = "missing_target"
all_rows["row_status"].value_counts()

## Keep the usable rows

In [ ]:
features = all_rows[all_rows["row_status"] == "kept"]
features = features.drop(columns="row_status")
features.head()

## Count the rows

In [ ]:
len(features)

## Count missing values by home

In [ ]:
counts = pd.crosstab(all_rows["LCLid"], all_rows["row_status"])
counts = counts.reindex(columns=["kept", "missing_target",
    "missing_lag_1h", "missing_lag_24h"], fill_value=0)
counts.columns = ["rows_kept", "excluded_missing_target",
    "excluded_missing_lag_1h", "excluded_missing_lag_24h"]
counts["grid_rows"] = all_rows.groupby("LCLid").size()
counts

## Count missing fields separately

In [ ]:
for field, name in [("target_kwh", "any_target_missing"),
                    ("lag_1h_kwh", "any_lag_1h_missing"), ("lag_24h_kwh", "any_lag_24h_missing")]:
    counts[name] = all_rows[field].isna().groupby(all_rows["LCLid"]).sum()
counts

## Count observed targets on the first day

In [ ]:
first_hour = all_rows.groupby("LCLid")["target_hour"].transform("min")
first_day = all_rows["target_hour"] < first_hour + pd.Timedelta(hours=24)
observed = first_day & all_rows["target_kwh"].notna()
counts["target_observed_in_first_24h"] = observed.groupby(all_rows["LCLid"]).sum()
counts

## Finish the count file

In [ ]:
column_order = ["grid_rows", "rows_kept", "excluded_missing_target", "excluded_missing_lag_1h",
                "excluded_missing_lag_24h", "any_target_missing", "any_lag_1h_missing",
                "any_lag_24h_missing", "target_observed_in_first_24h"]
counts = counts[column_order].copy()
counts.loc["TOTAL"] = counts.sum()
counts = counts.reset_index()
counts

## Make the examples easy to find

In [ ]:
examples = all_rows.copy()
examples["elapsed_hour"] = ((examples["target_hour"] - start) / pd.Timedelta(hours=1)).astype(int)
a = examples[examples["LCLid"] == "A"].set_index("elapsed_hour")
b = examples[examples["LCLid"] == "B"].set_index("elapsed_hour")
a.head()

## 1. Normal row: actual versus expected

In [ ]:
fields = ["target_kwh", "lag_1h_kwh", "lag_24h_kwh", "hour", "weekday"]
normal = pd.DataFrame({"actual": a.loc[30, fields], "expected": [3.0, 2.9, 0.6, 6, 2]})
normal["matches"] = np.isclose(normal["actual"].astype(float), normal["expected"])
normal

## 2. Never jump across an absent hour

In [ ]:
gap_examples = a.loc[[4, 5, 27, 28], ["lag_1h_kwh", "lag_24h_kwh", "row_status"]]
gap_examples

## 3. Unknown target stays unknown

In [ ]:
a.loc[26, ["target_kwh", "row_status"]]

## Check the kept table for gaps

In [ ]:
features.isna().any().any()

## 4. Unknown input stays unknown

In [ ]:
a.loc[27, ["lag_1h_kwh", "row_status"]]

## 5. Measured zero remains valid

In [ ]:
zero_values = pd.Series({"target_at_10": a.loc[10, "target_kwh"],
                         "lag_1h_at_11": a.loc[11, "lag_1h_kwh"],
                         "lag_24h_at_34": a.loc[34, "lag_24h_kwh"]})
zero_values, a.loc[34, "row_status"]

## 6. Homes do not share history

In [ ]:
b.loc[[0, 24], ["lag_1h_kwh", "lag_24h_kwh", "row_status"]]

## Check all kept histories by home

In [ ]:
(b.loc[b["row_status"] == "kept", "lag_1h_kwh"] >= 100).all(), (a.loc[a["row_status"] == "kept",
    "lag_1h_kwh"] < 10).all()

## 7. Independently calculate expected histories

In [ ]:
kept_a = a[a["row_status"] == "kept"]
i = kept_a.index.to_numpy()
expected = pd.DataFrame(index=kept_a.index)
expected["target_kwh"] = np.where(i == 10, 0, i / 10)
expected["lag_1h_kwh"] = np.where(i - 1 == 10, 0, (i - 1) / 10)
expected["lag_24h_kwh"] = np.where(i - 24 == 10, 0, (i - 24) / 10)
expected.head()

## Compare every kept A row

In [ ]:
history_matches = pd.DataFrame(np.isclose(kept_a[expected.columns], expected), columns=expected.columns)
history_matches.all()

## 8. Midnight and weekday

In [ ]:
a.loc[48, ["target_hour", "hour", "weekday", "lag_1h_kwh"]]

## 9. Counts agree

In [ ]:
counts[["LCLid", "grid_rows", "rows_kept", "any_target_missing"]]

## Compare the row counts

In [ ]:
count_fields = ["rows_kept", "excluded_missing_target",
    "excluded_missing_lag_1h", "excluded_missing_lag_24h"]
count_balance = counts[count_fields].sum(axis=1) == counts["grid_rows"]
count_balance

## 10. Detect duplicate keys

In [ ]:
duplicate_input = pd.concat([hourly, hourly.iloc[[5]]], ignore_index=True)
duplicate_rows = duplicate_input[duplicate_input.duplicated(["LCLid", "hour"], keep=False)]
duplicate_rows

## 11. Detect an off-hour timestamp

In [ ]:
off_hour_input = hourly.copy()
off_hour_input.loc[0, "hour"] += pd.Timedelta(minutes=30)
off_hour_rows = off_hour_input[off_hour_input["hour"] != off_hour_input["hour"].dt.floor("h")]
off_hour_rows

## 12. Save actual feature and count files

In [ ]:
from pathlib import Path
output_dir = PROJECT_ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

## Write both outputs

In [ ]:
feature_path = output_dir / "input_examples.csv"
counts_path = output_dir / "input_example_counts.csv"
features.to_csv(feature_path, index=False)
counts.to_csv(counts_path, index=False)
feature_path, counts_path

## Read the feature file back

In [ ]:
saved_features = pd.read_csv(feature_path)
saved_features["target_hour"] = pd.to_datetime(saved_features["target_hour"], utc=True)
saved_features.head()

## Compare the saved results

In [ ]:
saved_counts = pd.read_csv(counts_path)
same_keys = saved_features[["LCLid", "target_hour"]].equals(features[["LCLid",
    "target_hour"]].reset_index(drop=True))
same_values = np.allclose(saved_features[fields], features[fields], rtol=0, atol=1e-12)
list(saved_features.columns), same_keys, same_values, saved_features.isna().any().any(), saved_counts.equals(counts)